# Batch 1D / 2D Azimuthal Integration

Walk through the canonical batch-integration workflow in
`ssrl_xrd_tools`:

1. Load a pyFAI **PONI** calibration and a detector **mask**.
2. Iterate a directory of detector images and call
   `integrate_1d` / `integrate_2d` per frame.
3. Store the results to NeXus via `write_nexus` so they can be
   reloaded for analysis.
4. Quick sanity-check plot.

**Adapt the paths in the "Configuration" cell to your data.**


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted

from ssrl_xrd_tools.io import read_image, load_mask, write_nexus
from ssrl_xrd_tools.integrate import (
    load_poni,
    integrate_1d,
    integrate_2d,
)

## ✏️ Configuration

**Edit the cell below**, then run the rest of the notebook top-to-bottom.
Nothing in the later cells needs changing — the validation cell catches
typos before integration starts.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                  EDIT THIS CELL — paths + parameters                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── REQUIRED ───────────────────────────────────────────────────────────
data_dir   = Path('~/data/my_scan').expanduser()    # ← REPLACE
poni_file  = data_dir / 'calibration.poni'           # ← REPLACE
image_glob = '*.tif'                                  # pattern inside data_dir

# ── OPTIONAL paths ─────────────────────────────────────────────────────
mask_file  = data_dir / 'mask.edf'           # set to None if no mask file
out_file   = data_dir / 'integrated.nxs'

# ── Integration tuning ─────────────────────────────────────────────────
npt_1d  = 1000                # 1D radial bins
npt_2d  = (1000, 360)          # (radial, azimuthal) for 2D
unit    = 'q_A^-1'             # 'q_A^-1' | '2th_deg' | 'd_A'
method  = 'BBox'               # pyFAI integration method

### Validate the config

In [ ]:
assert data_dir.is_dir(), f'data_dir not found: {data_dir}'
assert poni_file.is_file(), f'poni_file not found: {poni_file}'
if mask_file is not None:
    assert mask_file.is_file(), f'mask_file not found: {mask_file}'
_images_preview = sorted(data_dir.glob(image_glob))
assert _images_preview, f'No images match {image_glob!r} in {data_dir}'
print(f'OK — {len(_images_preview)} image(s) found, PONI present.')

## Load calibration + mask

In [ ]:
poni = load_poni(poni_file)
mask = load_mask(mask_file) if mask_file and mask_file.exists() else None
print(f'PONI: dist={poni.dist:.4f} m, λ={poni.wavelength * 1e10:.4f} Å')
print(f'Mask: {mask.shape if mask is not None else "none"}')

## Discover images + batch integrate

In [ ]:
image_files = natsorted(data_dir.glob(image_glob))
print(f'Found {len(image_files)} images matching {image_glob!r}')

results_1d, results_2d = [], []
for i, path in enumerate(image_files):
    img = read_image(path, mask=mask)
    r1 = integrate_1d(img, poni, npt=npt_1d, unit=unit, method=method, mask=mask)
    r2 = integrate_2d(img, poni, npt=npt_2d, unit=unit, method=method, mask=mask)
    results_1d.append(r1)
    results_2d.append(r2)
    if (i + 1) % 10 == 0 or i == len(image_files) - 1:
        print(f'  integrated {i + 1}/{len(image_files)}')

## Save to NeXus

`write_nexus` writes a single self-describing .nxs file containing all
1D + 2D results with their axes, units, and metadata. Pass the result
collections as ``results_1d=`` / ``results_2d=`` dicts keyed by frame
index.

In [ ]:
results_1d_dict = {i: r for i, r in enumerate(results_1d)}
results_2d_dict = {i: r for i, r in enumerate(results_2d)}

write_nexus(
    str(out_file),
    results_1d=results_1d_dict,
    results_2d=results_2d_dict,
    overwrite=True,
)
print(f'Saved to {out_file}')

## Sanity-check: plot first frame

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

r1, r2 = results_1d[0], results_2d[0]
ax1.plot(r1.radial, r1.intensity)
ax1.set_xlabel(f'q ({r1.unit})')
ax1.set_ylabel('Intensity')
ax1.set_title('1D')

im = ax2.pcolormesh(r2.radial, r2.azimuthal, r2.intensity.T,
                    shading='auto', cmap='viridis')
ax2.set_xlabel(f'q ({r2.unit})')
ax2.set_ylabel('χ (deg)')
ax2.set_title('2D')
plt.colorbar(im, ax=ax2, label='Intensity')

plt.tight_layout()
plt.show()

---

### Next steps

- **Notebook 06 — Headless Reduction Pipeline** is the recommended
  higher-level alternative to the per-frame `integrate_1d` /
  `integrate_2d` loop above.  It uses `ReductionPlan` + `Scan` +
  `Frame` + `NexusSink` so the same configuration can drive notebook,
  CLI, and the xdart GUI without code changes.  Prefer it for
  new code; this notebook stays as the "low-level building blocks"
  reference.
- For multi-detector-angle scans where the detector sweeps and you want
  one stitched pattern: see notebook **02 — MultiGeometry stitching**.
- To do phase / peak fitting on the integrated results: see notebook
  **03 — Phase + peak fitting**.
